Əvvəl notebooku və SQL bağlantısını hazırlayırıq.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Layihənin əsas qovluğunu tapırıq
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

# SQL Server bağlantısı
engine = get_engine()

print("Silver mərhələsi üçün Python və SQL Server hazırdır!")

Silver mərhələsi üçün Python və SQL Server hazırdır!


Kredit məlumatlarını oxuyaq

In [2]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

print("Kredit sayı:", len(silver_loans_df))

print("\nSütunlar:")
print(silver_loans_df.columns.tolist())

display(silver_loans_df.head())

Kredit sayı: 4820

Sütunlar:
['loan_id', 'customer_id', 'application_date', 'product_type', 'amount', 'term_months', 'interest_rate', 'status', 'rejection_reason', 'defaulted', 'age', 'gender', 'city', 'branch', 'tenure_years', 'credit_score', 'balance', 'salary', 'products_count', 'has_card', 'is_active', 'churned', 'merge_date', 'is_holiday', 'holiday_name', 'is_weekend', 'is_workday', 'is_pre_holiday', 'temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max']


,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-01,1,Yeni il,0,0,0,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,Istehlak krediti,14942.94,15,13.74,R?dd edildi,Yüks?k borc yükü,NaN,...,1.0,2026-01-02,1,Yeni il,0,0,0,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,Istehlak krediti,24217.34,58,14.09,T?sdiql?ndi,NaN,0.0,...,1.0,2026-01-02,1,Yeni il,0,0,0,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,R?dd edildi,Asagi kredit skoru,NaN,...,1.0,2026-01-02,1,Yeni il,0,0,0,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,Istehlak krediti,6382.58,21,13.19,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-03,0,NaN,1,0,0,3.6,0.0,10.7


Təkrarlanan kreditləri yoxlayaq
təkrarlanan loan_id dəyərlərini yoxlayaq

In [3]:
# Təkrarlanan loan_id dəyərlərini tapırıq
duplicate_loans = silver_loans_df[
    silver_loans_df.duplicated(
        subset=["loan_id"],
        keep=False
    )
]

print("Ümumi kredit sətri:", len(silver_loans_df))

print(
    "Təkrarlanan loan_id sayı:",
    duplicate_loans["loan_id"].nunique()
)

print(
    "Təkrarlanan loan_id-lərə aid ümumi sətir sayı:",
    len(duplicate_loans)
)

display(
    duplicate_loans.sort_values("loan_id").head(10)
)

Ümumi kredit sətri: 4820
Təkrarlanan loan_id sayı: 15
Təkrarlanan loan_id-lərə aid ümumi sətir sayı: 30


,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,churned,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
227,L000228,C00112,1/12/2026,Avtomobil krediti,22141.14,53,10.40,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-12,0,NaN,0,1,0,7.1,0.0,14.3
228,L000228,C00112,1/12/2026,Avtomobil krediti,22141.14,53,10.40,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-12,0,NaN,0,1,0,7.1,0.0,14.3
492,L000492,C03849,1/21/2026,Istehlak krediti,2135.34,57,16.56,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-21,0,NaN,0,1,0,0.9,1.5,28.7
493,L000492,C03849,1/21/2026,Istehlak krediti,2135.34,57,16.56,T?sdiql?ndi,NaN,0.0,...,0.0,2026-01-21,0,NaN,0,1,0,0.9,1.5,28.7
946,L000945,C03694,2/3/2026,Istehlak krediti,1608.14,58,NaN,T?sdiql?ndi,NaN,0.0,...,1.0,2026-02-03,0,NaN,0,1,0,7.7,0.1,26.2
947,L000945,C03694,2/3/2026,Istehlak krediti,1608.14,58,NaN,T?sdiql?ndi,NaN,0.0,...,1.0,2026-02-03,0,NaN,0,1,0,7.7,0.1,26.2
1171,L001169,C04935,2/9/2026,Biznes krediti,40097.66,56,11.62,R?dd edildi,Qeyri-kafi g?lir s?n?di,NaN,...,0.0,2026-02-09,0,NaN,0,1,0,5.3,0.2,18.8
1172,L001169,C04935,2/9/2026,Biznes krediti,40097.66,56,11.62,R?dd edildi,Qeyri-kafi g?lir s?n?di,NaN,...,0.0,2026-02-09,0,NaN,0,1,0,5.3,0.2,18.8
1325,L001322,C03070,2/13/2026,Avtomobil krediti,11127.70,63,10.99,T?sdiql?ndi,NaN,0.0,...,0.0,2026-02-13,0,NaN,0,1,0,9.4,0.0,25.2
1326,L001322,C03070,2/13/2026,Avtomobil krediti,11127.70,63,10.99,T?sdiql?ndi,NaN,0.0,...,0.0,2026-02-13,0,NaN,0,1,0,9.4,0.0,25.2


Təkrarlanan kreditlər tam eynidirmi?

In [4]:
# Eyni loan_id olan sətirlərin bütün sütunlarını müqayisə edirik

loan_columns = [
    "loan_id",
    "customer_id",
    "application_date",
    "product_type",
    "amount",
    "term_months",
    "interest_rate",
    "status",
    "rejection_reason",
    "defaulted"
]

duplicate_original = silver_loans_df[
    silver_loans_df.duplicated(
        subset=["loan_id"],
        keep=False
    )
]

print(
    "Tamamilə eyni olan artıq sətirlərin sayı:",
    duplicate_original.duplicated(
        subset=loan_columns
    ).sum()
)

display(
    duplicate_original[
        loan_columns
    ].sort_values("loan_id").head(10)
)

Tamamilə eyni olan artıq sətirlərin sayı: 15


,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted
227,L000228,C00112,1/12/2026,Avtomobil krediti,22141.14,53,10.40,T?sdiql?ndi,NaN,0.0
228,L000228,C00112,1/12/2026,Avtomobil krediti,22141.14,53,10.40,T?sdiql?ndi,NaN,0.0
492,L000492,C03849,1/21/2026,Istehlak krediti,2135.34,57,16.56,T?sdiql?ndi,NaN,0.0
493,L000492,C03849,1/21/2026,Istehlak krediti,2135.34,57,16.56,T?sdiql?ndi,NaN,0.0
946,L000945,C03694,2/3/2026,Istehlak krediti,1608.14,58,NaN,T?sdiql?ndi,NaN,0.0
947,L000945,C03694,2/3/2026,Istehlak krediti,1608.14,58,NaN,T?sdiql?ndi,NaN,0.0
1171,L001169,C04935,2/9/2026,Biznes krediti,40097.66,56,11.62,R?dd edildi,Qeyri-kafi g?lir s?n?di,NaN
1172,L001169,C04935,2/9/2026,Biznes krediti,40097.66,56,11.62,R?dd edildi,Qeyri-kafi g?lir s?n?di,NaN
1325,L001322,C03070,2/13/2026,Avtomobil krediti,11127.70,63,10.99,T?sdiql?ndi,NaN,0.0
1326,L001322,C03070,2/13/2026,Avtomobil krediti,11127.70,63,10.99,T?sdiql?ndi,NaN,0.0


Təkrarlanan kreditləri silək

In [5]:
# Təkrarlanan kreditləri silirik
before_count = len(silver_loans_df)

silver_loans_df = silver_loans_df.drop_duplicates(
    subset="loan_id",
    keep="first"
).copy()

after_count = len(silver_loans_df)

print("Təmizləmədən əvvəl:", before_count)
print("Təmizləmədən sonra:", after_count)
print("Silinən təkrar sətirlər:", before_count - after_count)

print(
    "Qalan təkrarlanan loan_id sayı:",
    silver_loans_df["loan_id"].duplicated().sum()
)

Təmizləmədən əvvəl: 4820
Təmizləmədən sonra: 4805
Silinən təkrar sətirlər: 15
Qalan təkrarlanan loan_id sayı: 0


Boş (NaN) dəyərləri yoxlayaq

In [6]:
# Hər sütundakı boş dəyərlərin sayını hesablayırıq
missing_values = silver_loans_df.isna().sum()

# Yalnız boş dəyəri olan sütunları göstəririk
missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Boş dəyəri olan sütunlar:")
display(missing_values)

Boş dəyəri olan sütunlar:


holiday_name           4768
rejection_reason       3070
defaulted              1826
interest_rate            50
age                       5
city                      5
gender                    5
tenure_years              5
credit_score              5
balance                   5
branch                    5
salary                    5
products_count            5
is_active                 5
has_card                  5
churned                   5
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64